# sleep_level_events: records, coverage and defensible relationships
For Friday October 23, 2026. Only this table and the small/aggregated comparisons needed below are read, inside a read-only repeatable-read snapshot. No database changes are made. Outputs contain aggregate statistics and anonymous plots, not participant identifiers or raw records.

Shared IDs/group/type identify context; calendar/week/day label the export/reporting period; start/end/duration describe its window. Score definitions, label rules, timestamp conventions and device algorithms are unverified without the source dictionary. Names and SQL types are not definitions. No clinical thresholds, diagnosis, causation or predictive validation are used.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from sqlalchemy import create_engine, inspect, text
from dotenv import dotenv_values
from IPython.display import display, JSON, Markdown
PROJECT=Path.cwd()
url=dotenv_values(PROJECT/'.env').get('DATABASE_URL')
assert url, 'Configure DATABASE_URL privately in .env.'
engine=create_engine(url); del url
connection=engine.connect().execution_options(isolation_level='REPEATABLE READ')
transaction=connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
connection.execute(text("SET LOCAL statement_timeout = '240s'"))
def query(sql): return pd.read_sql(text(sql),connection)
key=['sdv_user_id','calendar_date','start_measure_period_sec','end_measure_period_sec']
results={}
def valid_key(df,k): return not df[k].isna().any(axis=None) and not df.duplicated(k).any()
def show_schema(table):
    cols=inspect(connection).get_columns(table)
    display(pd.DataFrame([{'column':c['name'],'SQL_type':str(c['type'])} for c in cols]))
    return [c['name'] for c in cols]
def audit_keys(df,candidates):
    display(pd.DataFrame([{'key':name,'null_key_rows':int(df[k].isna().any(axis=1).sum()),
      'rows_in_duplicate_keys':int(df.duplicated(k,keep=False).sum()),'unique_nonnull':valid_key(df,k)} for name,k in candidates.items()]))
def period_audit(p):
    p=p.copy(); p['calendar_date']=pd.to_datetime(p.calendar_date,errors='coerce')
    elapsed=p.end_measure_period_sec-p.start_measure_period_sec
    ordered=p.sort_values(key)
    prior=ordered.groupby('sdv_user_id').end_measure_period_sec.transform(lambda x:x.cummax().shift())
    utc=pd.to_datetime(p.start_measure_period_sec,unit='s',utc=True,errors='coerce')
    stats={'periods':len(p),'participants':p.sdv_user_id.nunique(),
      'null_key_rows':int(p[key].isna().any(axis=1).sum()),'duplicate_period_keys':int(p.duplicated(key,keep=False).sum()),
      'nonpositive_periods':int(elapsed.le(0).sum()),'exactly_24_hour_periods':int(elapsed.eq(86400).sum()),
      'overlapping_periods':int(ordered.start_measure_period_sec.lt(prior).sum()),
      'UTC_start_date_matches_label':int(utc.dt.tz_localize(None).dt.normalize().eq(p.calendar_date).sum())}
    if 'duration_measure_period_sec' in p:
        stats['duration_equals_end_minus_start']=int(p.duration_measure_period_sec.eq(elapsed).sum())
    display(pd.DataFrame([stats])); display((elapsed/3600).value_counts().sort_index().rename_axis('period_hours').to_frame('periods'))
    coverage=p.groupby('sdv_user_id').agg(first=('calendar_date','min'),last=('calendar_date','max'),observed_dates=('calendar_date','nunique'),periods=('calendar_date','size'))
    coverage['span_days']=(coverage['last']-coverage['first']).dt.days+1
    coverage['missing_dates_within_span']=coverage.span_days-coverage.observed_dates
    coverage['observed_date_percent']=100*coverage.observed_dates/coverage.span_days
    display(coverage[['observed_dates','periods','span_days','missing_dates_within_span','observed_date_percent']].describe().T)
    stats.update({'min_observed_dates_per_participant':int(coverage.observed_dates.min()),
      'max_observed_dates_per_participant':int(coverage.observed_dates.max()),
      'min_observed_date_percent':coverage.observed_date_percent.min(),'max_observed_date_percent':coverage.observed_date_percent.max()})
    return stats
def checked_join(left,right,label):
    if not (valid_key(left,key) and valid_key(right,key)):
        print(label+': skipped; keys are not unique and non-null.'); return None
    joined=left.merge(right,on=key,how='outer',suffixes=('_left','_right'),indicator=True,validate='one_to_one')
    counts={str(k):int(v) for k,v in joined._merge.value_counts().items()}
    results[label+'_period_join']=counts
    results[label+'_participant_overlap']={'left':left.sdv_user_id.nunique(),'right':right.sdv_user_id.nunique(),
      'shared':len(set(left.sdv_user_id.dropna()) & set(right.sdv_user_id.dropna()))}
    display(pd.Series(counts,name='periods').rename_axis(label+'_status').to_frame())
    return joined

TABLE='sleep_level_events'
columns=show_schema(TABLE)
d=query('SELECT * FROM '+TABLE)
d['calendar_date']=pd.to_datetime(d.calendar_date,errors='coerce')
results['inventory']={'rows':len(d),'participants':d.sdv_user_id.nunique(),'null_cells':int(d.isna().sum().sum()),'exact_duplicate_rows_after_first':int(d.duplicated().sum())}
display(pd.DataFrame([results['inventory']]))
display(pd.DataFrame({'column':d.columns,'pandas_type':d.dtypes.astype(str).values,'missing_rows':d.isna().sum().values,'distinct_nonnull':d.nunique().values}))


## 1. Event grain and candidate keys
A candidate row is a labelled interval inside an export record, not one whole day or one participant. Record IDs/period keys legitimately repeat across events. Test a full event-identity candidate (participant, date, period, start, derived end and label), timestamp collisions and exact row duplicates separately. Equal start times or intervals can represent different labels; no automatic deduplication is applied.

For body-battery events, end is operationally start + reported duration. For sleep-level events, duration is end − start. These use field-name seconds provisionally. Negative/zero duration, missing bounds and value candidates are explicit checks.

In [ ]:
LABEL='sleeptype'
d['_start']=d.start_sec; d['_end']=d.end_sec; d['_seconds']=d._end-d._start
event_key=key+['_start','_end',LABEL]
audit_keys(d,{'reporting_period_repeats_expected':key,'record_id_repeats_expected':['additional_id'],'event_identity_candidate':event_key,'participant_event_start':['sdv_user_id','_start']})
results['event_key_audit']={'null_event_key_rows':int(d[event_key].isna().any(axis=1).sum()),
 'duplicate_event_identity_rows':int(d.duplicated(event_key,keep=False).sum()),
 'participant_start_collision_rows':int(d.duplicated(['sdv_user_id','_start'],keep=False).sum())}
display(d[LABEL].fillna('<missing>').value_counts().rename_axis('recorded_label').to_frame('events'))
results['labels']={str(k):int(v) for k,v in d[LABEL].fillna('<missing>').value_counts().items()}
display(d._seconds.describe().to_frame('derived_event_seconds'))


## 2. Reporting periods and coverage
Reporting windows are checked separately from actual observations. End minus start, nonpositive windows, exact 24-hour windows and overlaps are diagnostics, not proof of wear or complete days. UTC conversion assumes Unix seconds only as a plausibility check; local calendar labels and inclusive/exclusive endpoints need documentation. Date coverage is unique observed dates divided by dates in each participant's first-to-last observed span; enrolment boundaries are unknown. Missing dates are never zero values. Participant identifiers are retained only in memory for grouping and are not displayed.

First aggregate events to one row per reporting period, retaining metadata-variation counts. Event absence does not mean no sleep/activity or a zero score. A reporting period can contain multiple intervals; a calendar label is not assumed to identify a complete sleep episode.

In [ ]:
p=d.groupby(key,dropna=False).agg(event_rows=('_start','size'),additional_id=('additional_id','first'),
 sdv_group_id=('sdv_group_id','first'),duration_measure_period_sec=('duration_measure_period_sec','first'),
 record_variants=('additional_id','nunique'),group_variants=('sdv_group_id','nunique'),duration_variants=('duration_measure_period_sec','nunique')).reset_index()
audit_keys(p,{'derived_period':key,'participant_date':['sdv_user_id','calendar_date']})
results['periods']=period_audit(p)
results['ambiguous_context_periods']=int(p[['record_variants','group_variants','duration_variants']].ne(1).any(axis=1).sum())
display(p.event_rows.describe().to_frame('events_per_period'))


## 3. Interval bounds, overlaps and union time
Use a declared **half-open [start,end)** interval convention for diagnostics; source endpoint rules are unknown. Count intervals outside the reporting window and UTC-date crossings separately. Cross-midnight is not an error, and UTC dates need not match the source's local labels.

For each period, compare summed valid event seconds with clipped interval-union seconds. The union counts overlapping time once, across labels, and clipping is used only for the explicit reporting-window diagnostic. Raw labelled durations remain separate. Sum minus union can reflect overlaps or duplicate exports; neither quantity is asserted to be official sleep time. Intervals fully outside a window contribute zero to its clipped union, but are still counted and described. Repeated intervals across periods are checked separately before any pooled total.

In [ ]:
valid=d._start.notna() & d._end.notna() & d._seconds.gt(0)
start_utc=pd.to_datetime(d._start,unit='s',utc=True,errors='coerce')
last_utc=pd.to_datetime(d._end-1,unit='s',utc=True,errors='coerce')
outside=valid & (d._start.lt(d.start_measure_period_sec) | d._end.gt(d.end_measure_period_sec))
results['interval_quality']={'missing_bound_rows':int((d._start.isna() | d._end.isna()).sum()),
 'nonpositive_duration_rows':int(d._seconds.le(0).sum()),'valid_intervals':int(valid.sum()),
 'outside_reporting_bounds':int(outside.sum()),
 'UTC_date_crossing_half_open_intervals':int((valid & start_utc.dt.normalize().ne(last_utc.dt.normalize())).sum()),
 'UTC_event_start_date_matches_label':int(start_utc.dt.tz_localize(None).dt.normalize().eq(d.calendar_date).sum())}
def union_seconds(intervals):
    total=0; previous_start=None; previous_end=None
    for a,b in sorted(intervals):
        if b<=a: continue
        if previous_end is None: previous_start,previous_end=a,b
        elif a>previous_end: total+=previous_end-previous_start; previous_start,previous_end=a,b
        else: previous_end=max(previous_end,b)
    if previous_end is not None: total+=previous_end-previous_start
    return total
rows=[]
for period,group in d.groupby(key,dropna=False):
    ws,we=period[-2:]
    if pd.isna(ws) or pd.isna(we) or we<=ws: continue
    events=group.loc[group._start.notna() & group._end.notna() & group._seconds.gt(0)]
    raw=list(zip(events._start,events._end))
    clipped=[(max(a,ws),min(b,we)) for a,b in raw if min(b,we)>max(a,ws)]
    clipped_sum=sum(b-a for a,b in clipped); unique=union_seconds(clipped)
    rows.append(dict(zip(key,period),raw_valid_event_seconds=sum(b-a for a,b in raw),
      summed_clipped_seconds=clipped_sum,union_clipped_seconds=unique,overlap_excess_seconds=clipped_sum-unique))
intervals=pd.DataFrame(rows)
p=p.merge(intervals,on=key,how='left',validate='one_to_one')
display(pd.DataFrame([results['interval_quality']]))
display(intervals[['raw_valid_event_seconds','summed_clipped_seconds','union_clipped_seconds','overlap_excess_seconds']].describe().T)
results['union_diagnostic']={'periods_with_clipped_overlap':int(intervals.overlap_excess_seconds.gt(0).sum()),
 'raw_valid_event_seconds':intervals.raw_valid_event_seconds.sum(),'summed_clipped_seconds':intervals.summed_clipped_seconds.sum(),
 'union_clipped_seconds_sum_across_periods':intervals.union_clipped_seconds.sum(),
 'overlap_excess_seconds_sum':intervals.overlap_excess_seconds.sum()}
# Count repeated participant/interval/label identities across reporting periods.
tokens=d[key].astype(str).agg('|'.join,axis=1)
membership=d.assign(_period_token=tokens).groupby(['sdv_user_id','_start','_end',LABEL],dropna=False)._period_token.nunique()
results['interval_quality']['interval_label_identities_spanning_periods']=int(membership.gt(1).sum())
conflicts=d.groupby(['sdv_user_id','_start','_end'],dropna=False)[LABEL].nunique()
results['interval_quality']['identical_interval_different_label_keys']=int(conflicts.gt(1).sum())
# Sweep raw valid intervals across all exports for each participant as a separate
# check: reporting-window clipping must not hide cross-period overlaps.
within_raw=0; across_raw=0; global_union=0
for _,group in d.loc[valid].groupby('sdv_user_id'):
    ordered=group.sort_values(['_start','_end'])
    active=[]
    for _,event in ordered.iterrows():
        a,b=event['_start'],event['_end']; token=tuple(event[k] for k in key[1:])
        active=[(end,owner) for end,owner in active if end>a]
        within_raw+=int(any(owner==token for _,owner in active))
        across_raw+=int(any(owner!=token for _,owner in active))
        active.append((b,token))
    global_union+=union_seconds(list(zip(ordered._start,ordered._end)))
results['raw_cross_period_overlap_diagnostic']={'rows_overlapping_prior_interval_same_period':within_raw,
 'rows_overlapping_prior_interval_other_period':across_raw,
 'union_seconds_across_all_exports_by_participant':global_union,
 'raw_sum_minus_global_union_seconds':d.loc[valid,'_seconds'].sum()-global_union}
display(pd.DataFrame([results['raw_cross_period_overlap_diagnostic']]))
display(d.loc[valid].groupby(LABEL)._seconds.agg(['count','median','min','max','sum']).rename(columns={'sum':'raw_labelled_seconds_can_overlap'}))
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
d[LABEL].value_counts().plot.barh(ax=axes[0]); axes[0].set(title='Recorded event labels',xlabel='Event rows')
(d.loc[valid,'_seconds']/3600).plot.hist(bins=25,ax=axes[1]); axes[1].set(title='Valid recorded interval lengths',xlabel='Hours (field-name interpretation)',ylabel='Events')
plt.tight_layout(); plt.show()


## 4. Sleep summaries and labelled event coverage
Aggregate event intervals to one row per exact reporting period before joining either sleep summary table. Five summary users versus eight event users is a coverage question; event-only periods do not imply zero score or no sleep. Check participant overlap, unmatched periods and shared context. Reporting periods are not automatically whole sleep episodes, and raw stage sums need not equal an official duration.

The `sleep` fields are categorical labels, so this notebook does not compare event seconds with purported numeric values in those fields. Rest/stage algorithms and label meanings remain unverified.

In [ ]:
for other in ['overall_sleep','sleep']:
    summary=query('SELECT * FROM '+other); summary['calendar_date']=pd.to_datetime(summary.calendar_date,errors='coerce')
    audit_keys(summary,{'period':key,'participant_date':['sdv_user_id','calendar_date']})
    j=checked_join(p,summary,other)
    if j is not None:
        both=j._merge.eq('both'); unique=both & j.record_variants.eq(1) & j.group_variants.eq(1) & j.duration_variants.eq(1)
        results[other+'_context']={field+'_equal':int((unique & j[field+'_left'].eq(j[field+'_right'])).sum()) for field in ['additional_id','sdv_group_id','duration_measure_period_sec']}
        display(j.groupby('_merge',observed=True).event_rows.agg(['count','median','min','max']))
fig,ax=plt.subplots(figsize=(8,3.5))
(p.union_clipped_seconds/3600).plot.hist(bins=25,ax=ax)
ax.set(title='Per-window union of clipped event intervals',xlabel='Union hours; not official sleep time',ylabel='Reporting periods')
plt.tight_layout(); plt.show()


## Verified findings and teacher takeaway

- **18,402 intervals, 797 reporting periods, eight participants**. No null cells, exact duplicates, nonpositive interval lengths, tested event-identity collisions or participant/start collisions occur. Period metadata are internally unique.
- Recorded labels are **light 8,767**, **rem 4,978**, **awake 2,593**, **deep 2,064**. These labels are described as recorded; stage-detection algorithms remain unverified.
- **1,864 intervals extend outside the labelled reporting window**, and **729 cross UTC dates** under the declared half-open convention. This requires date/window-assignment documentation, not automatic deletion or an assumption of error.
- Raw valid interval seconds total **23,189,062**. Clipping to reporting windows leaves **21,511,167 seconds**, with no clipped within-period overlap. A separate raw-interval sweep across all exports also finds **no within- or cross-period overlap** and no repeated interval/label identities across periods. The raw union equals the raw sum in this snapshot. These totals include **awake** labels and are **not official asleep duration**, wear time or a substitute for the categorical sleep-summary fields.
- **787 windows are exactly 24 hours**, ten are not; all durations equal end minus start, with no reporting-window overlap. Participants have **11–240 observed dates** and **58.8%–99.18% observed-date coverage** within their first-to-last spans, excluding unknown enrolment/wear boundaries.
- Each of `overall_sleep` and `sleep` matches **299 exact periods**, with **498 event-only periods** and no summary-only periods. Matched IDs/groups/durations agree. Event users number eight, summary users five, with all five shared. Events are aggregated before joining, so intervals cannot multiply summary rows.

**Takeaway:** “The event export covers more people and reporting periods than the summary tables. I checked actual intervals across midnight, reporting bounds and exports before calculating diagnostic unions. Event labels include awake, while the summary components are ratings, so neither is silently turned into an official total sleep duration.”


## Teacher-facing method and unresolved meanings
“I established whether rows were summaries or events, checked missing/duplicate/invalidity candidates, and validated period keys before any comparison. I kept unequal participant coverage and unmatched records visible.”

For events, sums can double-count overlapping intervals; interval unions use an explicit diagnostic convention. For sleep strings, labels are not numeric durations or percentages. Source score/impact/label definitions, algorithms, endpoints, timezone, enrolment and actual wear remain unresolved. Repeated records and unequal follow-up limit pooled interpretation. No medical, causal or predictive conclusion is made. CSV export, RAG and model building remain deferred.

In [ ]:
# End the read-only snapshot. No database changes were made.
transaction.rollback(); connection.close(); engine.dispose()
def scalar(v):
    if hasattr(v,'item'): return v.item()
    return str(v)
safe_results=json.loads(json.dumps(results,default=scalar))
display(JSON(safe_results))
print('All analysis cells completed; read-only connection closed.')
